# Notebook 06 — Database Vettoriali e Chunking

**Settimana:** 6
**Blocchi:** 4.2 Database vettoriali · 4.3 Chunking e indicizzazione
**Tempo stimato:** 3-4 ore

## Cosa costruirai

- Ingestione di un PDF reale con tre strategie di chunking diverse
- Indicizzazione in Chroma con metadati completi
- Confronto visivo: quali chunk recupera ogni strategia per le stesse query
- Ricerca con filtri sui metadati
- **Sfida finale:** indicizza tre PDF diversi in Chroma, cerca in tutti contemporaneamente e filtra per fonte

## Prerequisiti

```bash
pip install chromadb langchain langchain-text-splitters pymupdf openai python-dotenv
```

> **PDF per le esercitazioni:** puoi usare qualsiasi PDF in italiano (o altra lingua) che hai a disposizione.
> Se non ne hai uno, le celle includono un'alternativa che scarica
> il testo da Wikipedia al posto di un PDF.

---
## Cella 0 — Setup

In [ ]:
import os
import re
import json
import hashlib
import datetime
import numpy as np
import requests
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

load_dotenv()
OPENAI_KEY     = os.getenv("OPENAI_API_KEY")
cliente_openai = OpenAI(api_key=OPENAI_KEY)

# ── Funzione di embedding riutilizzata dalla settimana precedente ────────────
def generar_embeddings_batch(textos: list[str]) -> np.ndarray:
    respuesta = cliente_openai.embeddings.create(
        model="text-embedding-3-small",
        input=textos
    )
    vectores = [item.embedding for item in sorted(respuesta.data, key=lambda x: x.index)]
    return np.array(vectores)

def generar_embedding(texto: str) -> list[float]:
    r = cliente_openai.embeddings.create(model="text-embedding-3-small", input=texto)
    return r.data[0].embedding

# ── Directory di lavoro ────────────────────────────────────────────────────
Path("datos_pdf").mkdir(exist_ok=True)
Path("chroma_db").mkdir(exist_ok=True)

# ── Client Chroma persistente ────────────────────────────────────────────────
chroma_cliente = chromadb.PersistentClient(path="./chroma_db")
print("Chroma inizializzato. Collezioni esistenti:",
      [c.name for c in chroma_cliente.list_collections()])

---
## Cella 1 — Caricamento e pulizia del testo da PDF o Wikipedia

Carica il tuo PDF oppure usa l'alternativa di Wikipedia per procedere senza PDF.

In [ ]:
# ── Opzione A: carica un PDF tuo ─────────────────────────────────────────────

def cargar_pdf(ruta_pdf: str) -> list[dict]:
    """
    Estrae il testo da un PDF pagina per pagina con PyMuPDF.
    Restituisce una lista di dict: {texto, pagina, fuente}
    """
    import fitz   # PyMuPDF
    doc     = fitz.open(ruta_pdf)
    paginas = []
    for i, pagina in enumerate(doc):
        texto = pagina.get_text("text").strip()
        if texto:   # ignora le pagine vuote
            paginas.append({
                "texto":  texto,
                "pagina": i + 1,
                "fuente": Path(ruta_pdf).name
            })
    doc.close()
    return paginas


# ── Opzione B: scarica un articolo da Wikipedia (senza PDF) ─────────────────

def cargar_wikipedia(titulo: str, idioma: str = "it") -> list[dict]:
    """
    Scarica il testo completo di un articolo di Wikipedia.
    Simula le pagine dividendolo in blocchi di ~1500 caratteri.
    """
    url  = f"https://{idioma}.wikipedia.org/api/rest_v1/page/summary/{requests.utils.quote(titulo)}"
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()
    extracto = resp.json().get("extract", "")

    # Per più contenuto, usa l'API delle sezioni
    url_secciones = f"https://{idioma}.wikipedia.org/w/api.php"
    params = {
        "action": "query", "format": "json",
        "titles": titulo,  "prop": "extracts",
        "explaintext": True, "exsectionformat": "plain"
    }
    resp2  = requests.get(url_secciones, params=params, timeout=10)
    paginas_wiki = resp2.json().get("query", {}).get("pages", {})
    texto_completo = list(paginas_wiki.values())[0].get("extract", extracto)

    # Divide in blocchi simulando le pagine
    tamano_bloque = 1500
    bloques = []
    for i in range(0, len(texto_completo), tamano_bloque):
        bloque = texto_completo[i:i + tamano_bloque].strip()
        if bloque:
            bloques.append({
                "texto":  bloque,
                "pagina": i // tamano_bloque + 1,
                "fuente": f"wikipedia_{titulo.replace(' ', '_')}.txt"
            })
    return bloques


# ── Pulizia base del testo ───────────────────────────────────────────────────

def limpiar_texto(texto: str) -> str:
    """
    Pulizia base: normalizza gli spazi, elimina righe vuote multiple,
    elimina i caratteri di controllo.
    """
    texto = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", "", texto)  # caratteri di controllo
    texto = re.sub(r" +",  " ",  texto)    # spazi multipli
    texto = re.sub(r"\n{3,}", "\n\n", texto)  # più di 2 a capo
    return texto.strip()


# ── Carica il documento ──────────────────────────────────────────────────────
# OPZIONE A: inserisci il percorso al tuo PDF
# RUTA_PDF = "datos_pdf/tu_documento.pdf"
# paginas = cargar_pdf(RUTA_PDF)

# OPZIONE B: usa Wikipedia (decommenta se non hai un PDF)
TITULO_WIKIPEDIA = "Intelligenza artificiale"
paginas = cargar_wikipedia(TITULO_WIKIPEDIA)

# Pulisce il testo di ogni pagina
for p in paginas:
    p["texto"] = limpiar_texto(p["texto"])

# Testo completo concatenato (per alcune strategie di chunking)
texto_completo = "\n\n".join(p["texto"] for p in paginas)
fuente_nombre  = paginas[0]["fuente"] if paginas else "documento.txt"

print(f"Fonte:            {fuente_nombre}")
print(f"Pagine/blocchi:   {len(paginas)}")
print(f"Totale caratteri: {len(texto_completo):,}")
print(f"Totale parole:    {len(texto_completo.split()):,}")
print(f"\nPrimi 300 caratteri:\n{texto_completo[:300]}")

---
## Cella 2 — Le tre strategie di chunking a confronto

Applica le tre strategie allo stesso testo e osserva:
- quanti chunk produce ciascuna
- se i chunk rispettano i limiti naturali del testo
- la dimensione media di ogni chunk

In [ ]:
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    MarkdownHeaderTextSplitter,
)

# ── Strategia 1: dimensione fissa con sovrapposizione ────────────────────────

def chunking_fijo(
    texto: str,
    tamano: int = 500,
    solapamiento: int = 100
) -> list[str]:
    """Divide il testo ogni `tamano` caratteri con `solapamiento` di contesto."""
    chunks = []
    inicio = 0
    while inicio < len(texto):
        fin = min(inicio + tamano, len(texto))
        chunks.append(texto[inicio:fin])
        if fin == len(texto):
            break
        inicio += tamano - solapamiento
    return chunks


# ── Strategia 2: ricorsiva per separatori (LangChain) ─────────────────────────

splitter_recursivo = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", "! ", "? ", " ", ""],
    length_function=len,
)


# ── Strategia 3: chunking semantico ──────────────────────────────────────────
# Nota: questa strategia chiama l'API di embedding per ogni frase,
# quindi è più lenta e costosa. Qui usiamo una versione semplificata
# che raggruppa i paragrafi per similarità invece delle frasi.

def chunking_semantico_parrafos(
    texto: str,
    umbral_corte: float = 0.65,
    max_chars: int = 800
) -> list[str]:
    """
    Raggruppa paragrafi consecutivi che parlano dello stesso argomento.
    Taglia quando la similarità tra paragrafi consecutivi scende sotto la soglia.
    """
    parrafos = [p.strip() for p in texto.split("\n\n") if len(p.strip()) > 50]
    if len(parrafos) <= 1:
        return parrafos

    print(f"  Generazione embedding per {len(parrafos)} paragrafi...")
    embeddings_parrafos = generar_embeddings_batch(parrafos)

    chunks     = []
    chunk_actual = parrafos[0]

    for i in range(1, len(parrafos)):
        # Similarità tra il paragrafo attuale e quello precedente
        v_actual   = embeddings_parrafos[i]
        v_anterior = embeddings_parrafos[i - 1]
        norma_a    = np.linalg.norm(v_actual)
        norma_b    = np.linalg.norm(v_anterior)
        sim = float(np.dot(v_actual, v_anterior) / (norma_a * norma_b))

        chunk_potencial = chunk_actual + "\n\n" + parrafos[i]
        if sim < umbral_corte or len(chunk_potencial) > max_chars:
            chunks.append(chunk_actual)
            chunk_actual = parrafos[i]
        else:
            chunk_actual = chunk_potencial

    if chunk_actual:
        chunks.append(chunk_actual)

    return chunks


# ── Applica le tre strategie ──────────────────────────────────────────────────

print("Applicazione strategia 1: dimensione fissa...")
chunks_fijo      = chunking_fijo(texto_completo, tamano=500, solapamiento=100)

print("Applicazione strategia 2: ricorsiva...")
chunks_recursivo = splitter_recursivo.split_text(texto_completo)

print("Applicazione strategia 3: semantica...")
chunks_semantico = chunking_semantico_parrafos(texto_completo, umbral_corte=0.65)


# ── Statistiche comparative ───────────────────────────────────────────────────

def estadisticas_chunks(nombre: str, chunks: list[str]) -> None:
    tamanos = [len(c) for c in chunks]
    print(f"\n{'─'*50}")
    print(f"Strategia: {nombre}")
    print(f"  N. chunk:              {len(chunks)}")
    print(f"  Dimensione media:      {np.mean(tamanos):.0f} caratteri")
    print(f"  Dimensione min/max:    {min(tamanos)} / {max(tamanos)} caratteri")
    print(f"  Deviazione standard:   {np.std(tamanos):.0f} caratteri")
    # Mostra il primo chunk come esempio
    print(f"  Primo chunk:           {chunks[0][:120].replace(chr(10), ' ')}...")

estadisticas_chunks("Fisso (500/100)",     chunks_fijo)
estadisticas_chunks("Ricorsivo (500/100)", chunks_recursivo)
estadisticas_chunks("Semantico",           chunks_semantico)


# ── Ispezione visiva: dove taglia ogni strategia? ────────────────────────────
print("\n" + "="*60)
print("ISPEZIONE: i tagli avvengono in punti logici?")
print("\n[Fisso] Fine del chunk 0 / inizio del chunk 1:")
print(f"  ...{chunks_fijo[0][-80:]}")
print(f"  {chunks_fijo[1][:80]}...")

print("\n[Ricorsivo] Fine del chunk 0 / inizio del chunk 1:")
print(f"  ...{chunks_recursivo[0][-80:]}")
print(f"  {chunks_recursivo[1][:80]}...")

print("\nQuale rispetta meglio i limiti naturali del testo? → ...")

**Osservazioni** (modifica questa cella):

- Strategia che produce chunk più uniformi in dimensione: ...
- Strategia che rispetta meglio i limiti del testo: ...
- Il chunking semantico ha raggruppato i paragrafi in modo coerente?
- Per questo documento specifico, quale strategia sceglieresti in produzione e perché?

---
## Cella 3 — Indicizzazione in Chroma con metadati

Crea tre collezioni (una per strategia) e indicizza i chunk con metadati completi.
Osserva come l'aggiunta di metadati arricchisce ogni chunk senza aumentare il costo di embedding.

In [ ]:
def construir_metadatos(
    chunk: str,
    chunk_index: int,
    total_chunks: int,
    fuente: str,
    estrategia: str,
    pagina_aproximada: int | None = None
) -> dict:
    """Costruisce il dict di metadati per un chunk."""
    return {
        "fuente":          fuente,
        "estrategia":      estrategia,
        "chunk_index":     chunk_index,
        "total_chunks":    total_chunks,
        "pagina":          pagina_aproximada or -1,
        "n_chars":         len(chunk),
        "n_palabras":      len(chunk.split()),
        "fecha_ingesta":   datetime.date.today().isoformat(),
        "es_primer_chunk": chunk_index == 0,
        "es_ultimo_chunk": chunk_index == total_chunks - 1,
    }


def indexar_chunks(
    nombre_coleccion: str,
    chunks: list[str],
    fuente: str,
    estrategia: str,
    batch_size: int = 50
) -> chromadb.Collection:
    """
    Indicizza una lista di chunk in Chroma.
    Genera gli embedding in batch e aggiunge metadati a ogni chunk.
    Usa upsert per essere idempotente (sicuro da eseguire più volte).
    """
    # Elimina la collezione se esiste già, per ripartire pulito
    try:
        chroma_cliente.delete_collection(nombre_coleccion)
    except Exception:
        pass

    coleccion = chroma_cliente.create_collection(
        name=nombre_coleccion,
        metadata={"hnsw:space": "cosine"}
    )

    total = len(chunks)
    print(f"Indicizzazione di {total} chunk in '{nombre_coleccion}'...")

    for inicio in range(0, total, batch_size):
        fin       = min(inicio + batch_size, total)
        lote      = chunks[inicio:fin]

        # ID unici basati sull'hash del contenuto + indice
        ids = [
            f"{estrategia}_{hashlib.md5(c.encode()).hexdigest()[:8]}_{i}"
            for i, c in enumerate(lote, inicio)
        ]

        # Embedding in batch
        embeddings = generar_embeddings_batch(lote).tolist()

        # Metadati
        metadatos = [
            construir_metadatos(c, i, total, fuente, estrategia)
            for i, c in enumerate(lote, inicio)
        ]

        coleccion.upsert(
            ids=ids,
            documents=lote,
            embeddings=embeddings,
            metadatas=metadatos
        )

        print(f"  Lotto {inicio//batch_size + 1}: {fin}/{total} chunk indicizzati")

    print(f"✓ Collezione '{nombre_coleccion}': {coleccion.count()} documenti")
    return coleccion


# ── Indicizza le tre strategie ────────────────────────────────────────────────
col_fijo      = indexar_chunks("chunks_fijo",      chunks_fijo,      fuente_nombre, "fijo")
col_recursivo = indexar_chunks("chunks_recursivo", chunks_recursivo, fuente_nombre, "recursivo")
col_semantico = indexar_chunks("chunks_semantico", chunks_semantico, fuente_nombre, "semantico")

print("\nCollezioni disponibili in Chroma:")
for c in chroma_cliente.list_collections():
    col = chroma_cliente.get_collection(c.name)
    print(f"  {c.name}: {col.count()} documenti")

---
## Cella 4 — Confronto di retrieval: quali chunk recupera ogni strategia?

La prova reale: per le stesse query, quali frammenti recupera ogni strategia?
Sono ugualmente rilevanti? Quale offre più contesto utile?

In [ ]:
def buscar_en_coleccion(
    coleccion: chromadb.Collection,
    query: str,
    n_results: int = 3,
    filtro_metadatos: dict | None = None
) -> list[dict]:
    """
    Cerca in una collezione Chroma e restituisce i risultati normalizzati.
    Chroma restituisce la distanza, non la similarità — convertiamo: similarità = 1 - distanza.
    """
    vector_query = generar_embedding(query)

    kwargs = {
        "query_embeddings": [vector_query],
        "n_results":        min(n_results, coleccion.count()),
        "include":          ["documents", "distances", "metadatas"]
    }
    if filtro_metadatos:
        kwargs["where"] = filtro_metadatos

    resp = coleccion.query(**kwargs)

    resultados = []
    for doc, dist, meta in zip(
        resp["documents"][0],
        resp["distances"][0],
        resp["metadatas"][0]
    ):
        resultados.append({
            "texto":      doc,
            "similitud":  round(1 - dist, 4),   # Chroma cosine: distance = 1 - similarity
            "metadatos":  meta
        })
    return resultados


def comparar_estrategias(query: str, n_results: int = 2) -> None:
    """Mostra i risultati delle tre strategie per la stessa query."""
    print(f"\n{'='*70}")
    print(f"QUERY: '{query}'")

    for nombre, coleccion in [
        ("FISSO",     col_fijo),
        ("RICORSIVO", col_recursivo),
        ("SEMANTICO", col_semantico),
    ]:
        resultados = buscar_en_coleccion(coleccion, query, n_results)
        print(f"\n  [{nombre}]")
        for i, r in enumerate(resultados, 1):
            meta  = r["metadatos"]
            texto = r["texto"].replace("\n", " ")[:120]
            print(f"    {i}. score={r['similitud']:.4f} | "
                  f"chunk {meta['chunk_index']}/{meta['total_chunks']} | "
                  f"{meta['n_chars']} caratteri")
            print(f"       {texto}...")


# ── Query di confronto ────────────────────────────────────────────────────────
# Adatta le query in base al documento che hai caricato
QUERIES_COMPARATIVA = [
    "Quali sono le principali applicazioni dell'intelligenza artificiale?",
    "storia e origine dell'apprendimento automatico",
    "rischi e problemi etici dell'IA",
    "reti neurali profonde deep learning",
]

for query in QUERIES_COMPARATIVA:
    comparar_estrategias(query)

**Osservazioni** (modifica questa cella):

- Query in cui la strategia fissa è chiaramente inferiore: ...
- Ci sono query in cui il chunking semantico recupera chunk più coerenti? Quali?
- Quale score minimo ti sembra una soglia ragionevole per questo corpus?
- Se dovessi scegliere una strategia per la produzione con questo documento, quale sceglieresti e perché?

---
## Cella 5 — Ricerca con filtri sui metadati

I metadati non sono decorativi: permettono di restringere la ricerca prima di calcolare le similarità.
Prova i filtri e osserva come cambia il comportamento.

In [ ]:
# ── Esperimento 1: filtrare per posizione nel documento ──────────────────────
# Cerca solo nei primi chunk (introduzione) o negli ultimi (conclusione)

QUERY_FILTRO = "conclusioni e prospettive future"
print(f"Query: '{QUERY_FILTRO}'\n")

# Senza filtro
sin_filtro = buscar_en_coleccion(col_recursivo, QUERY_FILTRO, n_results=2)
print("Senza filtro:")
for r in sin_filtro:
    print(f"  chunk {r['metadatos']['chunk_index']} | score={r['similitud']:.4f} | "
          f"{r['texto'][:80].replace(chr(10), ' ')}...")

# Filtrando solo gli ultimi chunk (indice > 75% del totale)
total_chunks_col = col_recursivo.count()
indice_umbral    = int(total_chunks_col * 0.75)

# Nota: Chroma supporta gli operatori $gte, $lte, $eq, $ne, $in, $nin
con_filtro = buscar_en_coleccion(
    col_recursivo,
    QUERY_FILTRO,
    n_results=2,
    filtro_metadatos={"chunk_index": {"$gte": indice_umbral}}
)
print(f"\nSolo gli ultimi chunk (indice >= {indice_umbral}):")
for r in con_filtro:
    print(f"  chunk {r['metadatos']['chunk_index']} | score={r['similitud']:.4f} | "
          f"{r['texto'][:80].replace(chr(10), ' ')}...")


# ── Esperimento 2: filtrare per dimensione del chunk ──────────────────────────
print("\n" + "─"*60)
print("Solo chunk sostanziosi (> 300 caratteri):")

chunks_largos = buscar_en_coleccion(
    col_fijo,
    "apprendimento profondo reti neurali",
    n_results=3,
    filtro_metadatos={"n_chars": {"$gte": 300}}
)
for r in chunks_largos:
    m = r["metadatos"]
    print(f"  chunk {m['chunk_index']} | {m['n_chars']} caratteri | score={r['similitud']:.4f}")
    print(f"  {r['texto'][:100].replace(chr(10), ' ')}...")


# ── Esperimento 3: ispezione diretta dei metadati ────────────────────────────
print("\n" + "─"*60)
print("Metadati del primo chunk (collezione ricorsiva):")
primer_chunk = col_recursivo.get(
    where={"chunk_index": {"$eq": 0}},
    include=["documents", "metadatas"]
)
if primer_chunk["ids"]:
    print(json.dumps(primer_chunk["metadatas"][0], indent=2, ensure_ascii=False))

---
## Sfida finale — Multi-documento: indicizza tre fonti e cerca con filtri

Costruisci una collezione unificata con tre documenti diversi.
Implementa un motore di ricerca che permetta di:
1. Cercare in tutte le fonti contemporaneamente
2. Cercare solo in una fonte specifica
3. Cercare in più fonti contemporaneamente con un filtro `$in`

Leggi lo scheletro e completa i `# TODO`.

In [ ]:
# ── Passo 1: scarica tre documenti da Wikipedia (o usa i tuoi PDF) ───────────

DOCUMENTOS = [
    {"titulo": "Machine learning",              "tipo": "tecnologia"},
    {"titulo": "Elaborazione del linguaggio naturale", "tipo": "tecnologia"},
    {"titulo": "Etica dell'intelligenza artificiale",  "tipo": "sociale"},
]

print("Caricamento documenti...")
todos_los_chunks = []   # lista di dict: {texto, metadatos}

for doc in DOCUMENTOS:
    print(f"  Caricamento: {doc['titulo']}")
    paginas_doc = cargar_wikipedia(doc["titulo"])
    texto_doc   = "\n\n".join(limpiar_texto(p["texto"]) for p in paginas_doc)
    fuente_doc  = f"wikipedia_{doc['titulo'].replace(' ', '_')}.txt"

    chunks_doc = splitter_recursivo.split_text(texto_doc)
    for i, chunk in enumerate(chunks_doc):
        todos_los_chunks.append({
            "texto": chunk,
            "metadatos": {
                "fuente":        fuente_doc,
                "titulo":        doc["titulo"],
                "tipo":          doc["tipo"],
                "chunk_index":   i,
                "total_chunks":  len(chunks_doc),
                "n_chars":       len(chunk),
                "fecha_ingesta": datetime.date.today().isoformat(),
            }
        })
    print(f"    → {len(chunks_doc)} chunk")

print(f"\nTotale chunk multi-documento: {len(todos_los_chunks)}")


# ── Passo 2: indicizza tutto in una collezione unificata ──────────────────────

# TODO: crea una collezione chiamata "multi_doc" in Chroma
# e indicizza tutti i chunk di todos_los_chunks.
# Genera gli embedding in batch di 50 per non saturare l'API.
# Ogni documento deve avere un ID unico — puoi usare f"{fuente}_{chunk_index}".

def indexar_multidoc(chunks_con_meta: list[dict], nombre_col: str = "multi_doc") -> chromadb.Collection:
    """Indicizza una lista di {texto, metadatos} in Chroma."""
    try:
        chroma_cliente.delete_collection(nombre_col)
    except Exception:
        pass

    coleccion = chroma_cliente.create_collection(
        name=nombre_col,
        metadata={"hnsw:space": "cosine"}
    )

    # TODO: implementa l'indicizzazione in batch di 50
    # Suggerimento: itera da 0 a len(chunks_con_meta) con passo 50
    #      a ogni iterazione, estrai i testi, genera gli embedding,
    #      costruisci ID unici e chiama coleccion.upsert()

    raise NotImplementedError("Implementa l'indicizzazione")


# col_multidoc = indexar_multidoc(todos_los_chunks)


# ── Passo 3: implementa il motore di ricerca multi-fonte ──────────────────────

def buscar_multidoc(
    query: str,
    coleccion: chromadb.Collection,
    fuentes: list[str] | None = None,   # None = cerca in tutte
    tipo: str | None = None,            # filtra per tipo di documento
    n_results: int = 5
) -> list[dict]:
    """
    Motore di ricerca con filtri opzionali per fonte e tipo.

    Esempi d'uso:
    - buscar_multidoc("deep learning")                    # tutte le fonti
    - buscar_multidoc("deep learning", fuentes=["ML"])    # solo una fonte
    - buscar_multidoc("deep learning", tipo="tecnologia") # per tipo
    """
    # TODO: costruisci il filtro di metadati in base ai parametri
    # Suggerimento: se fuentes ha più di un elemento, usa {"fuente": {"$in": fuentes}}
    #      se fuentes ha un elemento solo, usa {"fuente": {"$eq": fuentes[0]}}
    #      combina i filtri con {"$and": [filtro1, filtro2]}
    raise NotImplementedError("Implementa il motore di ricerca")


# ── Passo 4: prova le tre modalità di ricerca ──────────────────────────────────
# Decommenta dopo aver implementato le funzioni precedenti

# QUERY = "regolamentazione etica e bias negli algoritmi di IA"

# print("=== Ricerca in TUTTE le fonti ===")
# for r in buscar_multidoc(QUERY, col_multidoc, n_results=4):
#     print(f"  [{r['metadatos']['titulo'][:30]}] {r['similitud']:.4f} | {r['texto'][:70]}...")

# print("\n=== Solo fonte: Etica dell'IA ===")
# fuente_etica = "wikipedia_Etica_dell'intelligenza_artificiale.txt"
# for r in buscar_multidoc(QUERY, col_multidoc, fuentes=[fuente_etica], n_results=3):
#     print(f"  {r['similitud']:.4f} | {r['texto'][:80]}...")

# print("\n=== Solo tipo: tecnologia ===")
# for r in buscar_multidoc(QUERY, col_multidoc, tipo="tecnologia", n_results=3):
#     print(f"  [{r['metadatos']['titulo'][:30]}] {r['similitud']:.4f} | {r['texto'][:70]}...")

### Estensioni facoltative

1. **Qdrant al posto di Chroma:** reimplementa la cella 3 usando Qdrant invece di Chroma. Confronta la sintassi dei filtri. Quale trovi più espressiva?

2. **Reindicizzazione incrementale:** scrivi una funzione `actualizar_fuente(ruta_pdf, coleccion)` che elimini tutti i chunk di quella fonte e li reindicizzi con il nuovo contenuto, senza toccare il resto.

3. **Statistiche dell'indice:** scrivi una funzione che mostri un riepilogo dell'indice: quanti chunk ci sono per fonte, quanti per data di ingestione, e qual è la distribuzione delle dimensioni.

In [ ]:
# Spazio per le estensioni facoltative

---
## Autovalutazione della settimana

1. **Cosa ha funzionato bene?**
   ...

2. **Cosa non hai capito del tutto?**
   ...

3. **Cosa cambieresti la settimana prossima?**
   ...

4. **Tempo reale dedicato:**
   ...

---
*Notebook 06 — Settimana 6 del Piano di Formazione AI Engineering*